In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
import os

In [5]:
load_dotenv()

True

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0,
    google_api_key=os.getenv("GEMINI_API_KEY")
)

In [7]:
# create a State

class LLMState(TypedDict):
    question: str
    answer : str

In [8]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # ask that question to the LLM
    answer = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer
    
    return state

In [9]:
# create our graph
graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)


# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

In [11]:
# execute

intial_state = {'question' : 'How far is moon from the earth?'}

final_state = workflow.invoke(intial_state)

print(final_state['answer'])

[{'type': 'text', 'text': 'On average, the Moon is about **238,855 miles (384,400 kilometers)** away from Earth. \n\nHowever, because the Moon travels in an elliptical (oval-shaped) orbit rather than a perfect circle, the distance changes constantly:\n\n* **Closest approach (Perigee):** About **225,623 miles** (363,300 km)\n* **Farthest distance (Apogee):** About **251,966 miles** (405,500 km)\n\n**To put that in perspective:** You could fit about 30 planet Earths side-by-side in the space between the Earth and the Moon.', 'extras': {'signature': 'EtkOCtYOAWkUfRO7jDaUsBkruvwpXWcF0UqLgE+qjUCjCa0LLBeo992qY41pMr/VDr9l+rXNy08o3DhHK9ZquvYdKF1UammZJQ0pG1uk0Fl8r53ishcKW6TCazlxMYud9NNmmGGnB9jYTF1TsQAiuH4TYqkERDHoz3oTm+2NubCmR3lmMKMIxcnSmqA6exK9x8h5hY/PhmAySNqYQIqJahi8SJfSBVjHi2qOPu5tB7eHa/yxfRzbimo6txDXdP5iu6ZeSj1vc00n370tEPXhWaDJtgtrHKZRfOOuKYwj8FzITTGPgHlEtui2mKvrHL4YbfH2tCSTr1Ak9kJxhLw0mC06O5210ghU+jf1WwgKqEaA+yPr9Q5TaD9zLepZaeZa4iCgQB0kQ7FdmpOzx0XyXZzACGslEiHFgELqV4CPOTZuUZXqCXzzIAlsuVKV85